# Phase 1 — Parse-level cleaning

This notebook applies all required parse-time cleaning steps before any EDA.


In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

raw_path = "dcs_student_data.csv"
df = pd.read_csv(raw_path)

print(f"Raw shape: {df.shape}")
df.head()


Raw shape: (10030, 21)


,Student_ID,First_Name,Last_Name,Email,Gender,Age,Department,Attendance (%),Midterm_Score,Final_Score,Assignments_Avg,Quizzes_Avg,Participation_Score,Projects_Score,Total_Score,Grade,test_preparation_course,math_score,reading_score,writing_score,science_score
0,S1000,Omar,Williams,student0@university.com,Female,22.0,Engineering,52.29,55.03,57.82,84.22,74.06,3.99,85.90,56.09,F,1,89,38.0,85.0,26.0
1,S1001,Maria,Brown,student1@university.com,Male,18.0,Engineering,97.27,97.23,45.80,74.32,94.24,8.32,55.65,50.64,A,0,65,100.0,67.0,96.0
2,S1002,Ahmed,Jones,student2@university.com,Male,24.0,Business,57.19,67.05,93.68,67.70,85.70,5.05,73.79,70.30,D,0,10,99.0,97.0,58.0
3,S1003,Omar,Williams,student3@university.com,Female,24.0,Mathematics,95.15,47.79,80.63,66.06,93.51,6.54,92.12,61.63,A,1,22,51.0,41.0,84.0
4,S1004,John,Smith,student4@university.com,Female,23.0,CS,54.18,46.59,78.89,96.85,83.70,5.97,68.42,66.13,F,1,26,58.0,64.0,65.0


In [2]:
score_columns = [
    "Attendance (%)",
    "Midterm_Score",
    "Final_Score",
    "Assignments_Avg",
    "Quizzes_Avg",
    "Participation_Score",
    "Projects_Score",
    "Total_Score",
    "math_score",
    "reading_score",
    "writing_score",
    "science_score",
]

# Coerce score columns and Age to numeric (non-numeric values become NaN)
for col in score_columns + ["Age"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Normalize categorical text and merge synonyms
df["Gender"] = df["Gender"].astype(str).str.strip().str.lower().map({
    "female": "Female",
    "male": "Male",
}).fillna(df["Gender"].astype(str).str.strip().str.title())

dept_map = {
    "engineering": "Engineering",
    "business": "Business",
    "mathematics": "Mathematics",
    "math": "Mathematics",
    "computer science": "Computer Science",
    "cs": "Computer Science",
}
df["Department"] = (
    df["Department"].astype(str).str.strip().str.lower().map(dept_map)
    .fillna(df["Department"].astype(str).str.strip().str.title())
)

# Apply invalid-value rules
for col in score_columns:
    df.loc[(df[col] < 0) | (df[col] > 100), col] = np.nan

df.loc[(df["Age"] < 15) | (df["Age"] > 60), "Age"] = np.nan

# Drop duplicate Student_ID rows, keep first occurrence
rows_before = len(df)
df = df.drop_duplicates(subset=["Student_ID"], keep="first").copy()
rows_after = len(df)

print(f"Rows before duplicate removal: {rows_before}")
print(f"Rows after duplicate removal:  {rows_after}")
print(f"Duplicates removed:            {rows_before - rows_after}")


Rows before duplicate removal: 10030
Rows after duplicate removal:  10000
Duplicates removed:            30


In [3]:
missing_summary = df[["Age"] + score_columns].isna().sum().sort_values(ascending=False)
print("Missing values after cleaning:")
print(missing_summary)

print("\nNormalized category values:")
print("Gender:", sorted(df["Gender"].dropna().unique().tolist()))
print("Department:", sorted(df["Department"].dropna().unique().tolist()))


Missing values after cleaning:
Age                    202
Attendance (%)         202
Midterm_Score          202
Final_Score            202
Assignments_Avg        200
Quizzes_Avg            200
Participation_Score    200
Projects_Score         200
math_score               1
Total_Score              0
reading_score            0
writing_score            0
science_score            0
dtype: int64

Normalized category values:
Gender: ['Female', 'Male']
Department: ['Business', 'Computer Science', 'Engineering', 'Mathematics']


In [4]:
clean_path = "dcs_student_data_cleaned.csv"
df.to_csv(clean_path, index=False)
print(f"Saved cleaned dataset to: {clean_path}")


Saved cleaned dataset to: dcs_student_data_cleaned.csv


Phase 1 is complete: numeric coercion, categorical normalization, invalid-value handling, and duplicate removal have been applied.
